## Girilen Bir Bibere En Uygun Eşi Bulma

### Kütüphane ve Veriyi Yükleme

In [17]:
import pandas as pd
import os


veri_dosyasi = "veri.xlsx"
os.chdir(os.path.dirname(veri_dosyasi))


df = pd.read_excel(veri_dosyasi, sheet_name="Sayfa1")

### Temizlik, Mendel Olasılıkları Tanımları ve Ağırlık Seçtirme

In [18]:
df = df[['Field Nr', 'Tm0(L1)', 'Tm2(L3)', 'Tswv']]

df = df.dropna()
df = df[~df[['Tm0(L1)', 'Tm2(L3)', 'Tswv']].isin(['NTC', '?']).any(axis=1)]


df.reset_index(drop=True, inplace=True)


mendel_probs = {
    ('R', 'R'): {'R': 1.0, 'HT': 0.0, 'S': 0.0},
    ('R', 'HT'): {'R': 0.5, 'HT': 0.5, 'S': 0.0},
    ('HT', 'R'): {'R': 0.5, 'HT': 0.5, 'S': 0.0},
    ('R', 'S'): {'R': 0.0, 'HT': 1.0, 'S': 0.0},
    ('S', 'R'): {'R': 0.0, 'HT': 1.0, 'S': 0.0},
    ('HT', 'HT'): {'R': 0.25, 'HT': 0.5, 'S': 0.25},
    ('HT', 'S'): {'R': 0.0, 'HT': 0.5, 'S': 0.5},
    ('S', 'HT'): {'R': 0.0, 'HT': 0.5, 'S': 0.5},
    ('S', 'S'): {'R': 0.0, 'HT': 0.0, 'S': 1.0}
}

print("Skorlama yöntemini seçin:")
print("1 - HT = R = 1.0")
print("2 - HT = 1.0, R = 0.7")
secim = input("Seçiminiz (1/2): ")

if secim == "1":
    HT_SKOR, R_SKOR = 1.0, 1.0
elif secim == "2":
    HT_SKOR, R_SKOR = 1.0, 0.7
else:
    print("Geçersiz giriş. Varsayılan: HT = R = 1.0")
    HT_SKOR, R_SKOR = 1.0, 1.0


Skorlama yöntemini seçin:
1 - HT = R = 1.0
2 - HT = 1.0, R = 0.7


### Anne Yaprağın Hastalıklara Karşı Direnç Bilgisi(Tm0,Tm2,Tswv)

#### Boş bırakılabilir bir filtreleme görevi görür

In [19]:
print("Anne filtresi için genetik değerleri girin (boş bırakabilirsiniz):")
anne_tm0 = input("Tm0 geni (R / HT / S veya boş): ").strip().upper()
anne_tm2 = input("Tm2 geni (R / HT / S veya boş): ").strip().upper()
anne_tswv = input("Tswv geni (R / HT / S veya boş): ").strip().upper()

anne_df = df.copy()
if anne_tm0 in ['R', 'HT', 'S']:
    anne_df = anne_df[anne_df['Tm0(L1)'] == anne_tm0]
if anne_tm2 in ['R', 'HT', 'S']:
    anne_df = anne_df[anne_df['Tm2(L3)'] == anne_tm2]
if anne_tswv in ['R', 'HT', 'S']:
    anne_df = anne_df[anne_df['Tswv'] == anne_tswv]


Anne filtresi için genetik değerleri girin (boş bırakabilirsiniz):


### Veri Setindeki Tüm Yapraklarla Çaprazlanması ve Çocuğun Direnç Skorlarının Belirlenmesi

In [20]:
if anne_df.empty:
    print("\n Seçilen filtrelere uyan hiçbir anne adayı bulunamadı.")
else:
    for _, anne in anne_df.iterrows():
        anne_id = int(anne['Field Nr'])
        print(f"\n🔬 Anne: {anne_id} - Tm0: {anne['Tm0(L1)']}, Tm2: {anne['Tm2(L3)']}, Tswv: {anne['Tswv']}")

        adaylar = []
        for _, baba in df.iterrows():
            if baba['Field Nr'] == anne_id:
                continue  # Kendisiyle çaprazlama olmasın

            total_r = 0
            prob_text = {}

            for gen, col in zip(['Tm0', 'Tm2', 'Tswv'], ['Tm0(L1)', 'Tm2(L3)', 'Tswv']):
                g1 = anne[col]
                g2 = baba[col]
                probs = mendel_probs.get((g1, g2), {'R': 0.0, 'HT': 0.0, 'S': 0.0})
                total_r += probs['R'] * R_SKOR + probs['HT'] * HT_SKOR

                prob_text[gen] = f"%{int(probs['R']*100)} R, %{int(probs['HT']*100)} HT, %{int(probs['S']*100)} S"
          
            adaylar.append({
                'Baba': int(baba['Field Nr']),
                'Baba_Tm0': baba['Tm0(L1)'],
                'Baba_Tm2': baba['Tm2(L3)'],
                'Baba_Tswv': baba['Tswv'],
                'Tm0_prob': prob_text['Tm0'],
                'Tm2_prob': prob_text['Tm2'],
                'Tswv_prob': prob_text['Tswv'],
                'Toplam_R_Skoru': total_r
            })
            
    

        # ✅ En iyi 5 baba adayı
        en_iyi = sorted(adaylar, key=lambda x: x['Toplam_R_Skoru'], reverse=True)[:5]
        for i, b in enumerate(en_iyi, 1):
            print(f"{i}. Baba: {b['Baba']} - Tm0: {b['Baba_Tm0']}, Tm2: {b['Baba_Tm2']}, Tswv: {b['Baba_Tswv']}")
            print(f"   Çocuk Genotipi Olasılıkları: Tm0: {b['Tm0_prob']}, Tm2: {b['Tm2_prob']}, Tswv: {b['Tswv_prob']}")
            print(f"   Toplam Direnç Skoru: {b['Toplam_R_Skoru']:.2f}\n")



🔬 Anne: 230 - Tm0: R, Tm2: HT, Tswv: HT
1. Baba: 11 - Tm0: S, Tm2: R, Tswv: R
   Çocuk Genotipi Olasılıkları: Tm0: %0 R, %100 HT, %0 S, Tm2: %50 R, %50 HT, %0 S, Tswv: %50 R, %50 HT, %0 S
   Toplam Direnç Skoru: 3.00

2. Baba: 46 - Tm0: S, Tm2: R, Tswv: R
   Çocuk Genotipi Olasılıkları: Tm0: %0 R, %100 HT, %0 S, Tm2: %50 R, %50 HT, %0 S, Tswv: %50 R, %50 HT, %0 S
   Toplam Direnç Skoru: 3.00

3. Baba: 47 - Tm0: S, Tm2: R, Tswv: R
   Çocuk Genotipi Olasılıkları: Tm0: %0 R, %100 HT, %0 S, Tm2: %50 R, %50 HT, %0 S, Tswv: %50 R, %50 HT, %0 S
   Toplam Direnç Skoru: 3.00

4. Baba: 48 - Tm0: S, Tm2: R, Tswv: R
   Çocuk Genotipi Olasılıkları: Tm0: %0 R, %100 HT, %0 S, Tm2: %50 R, %50 HT, %0 S, Tswv: %50 R, %50 HT, %0 S
   Toplam Direnç Skoru: 3.00

5. Baba: 49 - Tm0: S, Tm2: R, Tswv: R
   Çocuk Genotipi Olasılıkları: Tm0: %0 R, %100 HT, %0 S, Tm2: %50 R, %50 HT, %0 S, Tswv: %50 R, %50 HT, %0 S
   Toplam Direnç Skoru: 3.00

